# 01 — Explore Legacy Source
Profile source tables before migration: row counts, null rates, date ranges.

In [ ]:
import sys; sys.path.insert(0, '..')
from src.utils.spark_session import get_spark
spark = get_spark('../config/env.yaml')
spark

In [ ]:
import yaml
with open('../config/env.yaml') as f:
    cfg = yaml.safe_load(f)

src = cfg['source']
for tbl in cfg['tables']:
    df = (spark.read.format('jdbc')
          .option('url', src['url'])
          .option('dbtable', tbl['source_table'])
          .option('user', src['user'])
          .option('password', src['password'])
          .load())
    print(f"\n=== {tbl['source_table']} ===")
    print(f"  rows: {df.count():,}")
    df.printSchema()
    df.describe().show()

## Null rate per column

In [ ]:
from pyspark.sql import functions as F

# Edit table name as needed
df = spark.read.format('jdbc').option('url', src['url']).option('dbtable', 'public.transactions').option('user', src['user']).option('password', src['password']).load()
total = df.count()
null_rates = [(c, df.filter(F.col(c).isNull()).count() / total) for c in df.columns]
for col, rate in sorted(null_rates, key=lambda x: -x[1]):
    if rate > 0:
        print(f'  {col}: {rate:.1%}')